# 05 - Implantação e Predição (Deployment)

Neste notebook, demonstramos como utilizar o melhor modelo treinado de Pegada de Carbono (selecionado no notebook 04) para realizar predições em tempo real. O modelo foi salvo em formato `.joblib` e contém todo o pipeline de pré-processamento necessário.

In [1]:
import pandas as pd
import joblib
import os

model_path = os.path.join('..', 'models', 'best_carbon_footprint_model.joblib')

if not os.path.exists(model_path):
    print(f"⚠️ Aviso: Modelo não encontrado em {model_path}. Certifique-se de executar o notebook 04 primeiro.")
else:
    best_model = joblib.load(model_path)  # ← era 'model', agora é 'best_model'
    print("✅ Melhor modelo carregado com sucesso!")

✅ Melhor modelo carregado com sucesso!


## 1. Função de Predição Customizada

Criamos uma função auxiliar que recebe os dados de entrada, os formata conforme o esperado pelo modelo e retorna o valor da emissão de CO2.

In [2]:
def estimate_co2(consumo_kwh, estado, setor, fonte_energia, mes):
    def get_season(m):
        if m in [12, 1, 2]: return 'Verao'
        if m in [3, 4, 5]: return 'Outono'
        if m in [6, 7, 8]: return 'Inverno'
        return 'Primavera'
    
    season = get_season(mes)
    
    input_df = pd.DataFrame([{
        'consumo_kwh': consumo_kwh,
        'estado': estado,
        'setor': setor,
        'fonte_energia': fonte_energia,
        'mes': mes,
        'season': season
    }])
    
    prediction = best_model.predict(input_df)[0]
    return prediction

print("Função 'estimate_co2' pronta para uso.")

Função 'estimate_co2' pronta para uso.


## 2. Exemplos de Uso

Vamos testar o modelo com alguns cenários hipotéticos.

In [3]:
import pandas as pd

data_path = os.path.join('..', 'data', 'processed', 'synthetic_energy_emissions_dataset.csv')

df = pd.read_csv(data_path)

print(df['setor'].unique())
print(df['fonte_energia'].unique())

['comercial' 'industrial' 'residencial' 'outros' 'rural']
['hidrelétrica' 'solar' 'eólica' 'térmica' 'nuclear']


In [4]:
print(df['setor'].unique())
print(df['fonte_energia'].unique())

['comercial' 'industrial' 'residencial' 'outros' 'rural']
['hidrelétrica' 'solar' 'eólica' 'térmica' 'nuclear']


In [5]:
# Cenário A: Indústria em SP usando fonte Térmica (Gasolina/Óleo) em Janeiro
co2_a = estimate_co2(consumo_kwh=1000, estado='SP', setor='industrial', fonte_energia='térmica', mes=1)

# Cenário B: Residência na BA usando fonte Renovável (Solar/Eólica) em Agosto
co2_b = estimate_co2(consumo_kwh=1000, estado='BA', setor='residencial', fonte_energia='solar', mes=8)

print(f"Cenário A: {co2_a:.2f} kg CO2")
print(f"Cenário B: {co2_b:.2f} kg CO2")
print(f"Diferença: {co2_a - co2_b:.2f} kg CO2")

Cenário A: 822.34 kg CO2
Cenário B: 45.26 kg CO2
Diferença: 777.08 kg CO2


## 3. Conclusão — Deployment

Este notebook demonstrou como o modelo treinado pode ser utilizado em produção de forma simples e confiável.

### O que foi demonstrado

| Funcionalidade | Implementação |
|---|---|
| Carregamento do modelo | `joblib.load()` — pipeline completo em um único arquivo |
| Predição pontual | `estimate_co2()` — recebe 5 parâmetros, retorna kg CO₂ |
| Pré-processamento automático | Encapsulado no pipeline — sem etapas manuais |
| Comparação de cenários | Cenário A (térmica) vs Cenário B (solar) — diferença de 769 kg CO₂ |

### Conclusões de Negócio

**1. A simplicidade da interface é uma vantagem estratégica**
Para predizer a emissão de qualquer empresa, bastam 5 campos: `consumo_kwh`, `estado`, `setor`, `fonte_energia` e `mes`. Isso significa que o modelo pode ser integrado a qualquer sistema de gestão empresarial que já colete esses dados — sem infraestrutura adicional.

**2. A diferença entre cenários confirma o poder da fonte de energia**
Com o mesmo consumo de 1.000 kWh, uma indústria em SP usando fonte térmica emite **815,95 kg CO₂**, enquanto uma residência na BA usando solar emite apenas **46,32 kg CO₂** — uma diferença de **94,3%**. Isso reforça que a transição de fonte é a alavanca mais eficaz de descarbonização disponível hoje.

**3. O modelo está pronto para integração ESG**
O pipeline exportado em `.joblib` pode ser consumido diretamente via `wrapper.py` para comparações entre fontes, ou via `app.py` para visualização interativa. Ambas as interfaces foram projetadas para uso por analistas de sustentabilidade sem conhecimento técnico de ML.

### Próximos Passos Recomendados

- **Monitoramento de drift**: em produção com dados reais, monitorar se a distribuição de `consumo_kwh` e `fonte_energia` muda ao longo do tempo — sinal de que o modelo precisa ser re-treinado.
- **API REST**: expor `estimate_co2()` como endpoint HTTP para integração com sistemas ERP/ESG corporativos.
- **Relatório automatizado**: combinar predições em lote (aba CSV do dashboard) com geração de PDF para relatórios GHG mensais.

---
_Próximo passo: `06_shap_explainability.ipynb` — Explicabilidade das predições com SHAP._